# Asset-Backed Securities (ABS) Pricing Model
## Residential Mortgage-Backed Security (RMBS) — Sequential Pay Structure

**Methodology:**
- PSA prepayment model (Public Securities Association convention)
- CDR default model with loss severity
- Three-tranche sequential pay waterfall (Senior / Mezzanine / Equity)
- Live FRED discount curve (falls back to hardcoded curve if offline)
- Sensitivity analysis: price and WAL across CPR multiples and default rates

**Pool:** Hypothetical $100M 30-year fixed-rate mortgage pool (standard academic construct)

**Discount rates:** Live US Treasury yields from FRED (same data source as Hull-White model)

In [1]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')

try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    REQUESTS_AVAILABLE = False

## 1. Live Discount Curve — FRED US Treasury Yields
Fetches live constant-maturity Treasury yields from FRED. Falls back to hardcoded curve (Mar 2025) if offline.

In [2]:
def fetch_fred_series(series_id):
    """Fetch a single FRED series as DataFrame."""
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    df = pd.read_csv(url)
    df.columns = ["Date", series_id]
    df["Date"] = pd.to_datetime(df["Date"])
    df[series_id] = pd.to_numeric(df[series_id], errors="coerce")
    return df


def fetch_live_curve():
    """Build full Treasury curve from FRED. Returns (date, curve_df)."""
    series = {
        "DGS1MO":  1/12,
        "DGS3MO":  0.25,
        "DGS6MO":  0.50,
        "DGS1":    1.00,
        "DGS2":    2.00,
        "DGS3":    3.00,
        "DGS5":    5.00,
        "DGS7":    7.00,
        "DGS10":  10.00,
        "DGS20":  20.00,
        "DGS30":  30.00,
    }
    dfs = [fetch_fred_series(sid) for sid in series]
    merged = dfs[0]
    for d in dfs[1:]:
        merged = merged.merge(d, on="Date", how="outer")
    merged = merged.sort_values("Date").reset_index(drop=True)
    valid = merged.dropna(subset=list(series.keys()), how="all")
    row = valid.iloc[-1]
    selected_date = row["Date"]
    pts = []
    for sid, T in series.items():
        y_pct = row[sid]
        if pd.isna(y_pct):
            continue
        y = float(y_pct) / 100.0
        P0T = np.exp(-y * T)
        pts.append((T, y, P0T))
    curve = pd.DataFrame(pts, columns=["T_years", "y_decimal", "P0T"])
    return selected_date, curve


def fallback_curve():
    """Hardcoded US Treasury curve — March 2025 (offline fallback)."""
    data = [
        (1/12,  0.0528),
        (0.25,  0.0524),
        (0.50,  0.0516),
        (1.00,  0.0500),
        (2.00,  0.0399),  # Source: US Treasury, March 2025
        (3.00,  0.0392),
        (5.00,  0.0400),
        (7.00,  0.0416),
        (10.00, 0.0431),
        (20.00, 0.0473),
        (30.00, 0.0463),
    ]
    pts = [(T, y, np.exp(-y * T)) for T, y in data]
    return "2025-03-21 (hardcoded fallback)", pd.DataFrame(pts, columns=["T_years", "y_decimal", "P0T"])


# Attempt live fetch; fall back gracefully
if REQUESTS_AVAILABLE:
    try:
        curve_date, curve = fetch_live_curve()
        data_source = "FRED (live)"
    except Exception as e:
        curve_date, curve = fallback_curve()
        data_source = f"Hardcoded fallback (FRED unavailable: {type(e).__name__})"
else:
    curve_date, curve = fallback_curve()
    data_source = "Hardcoded fallback (requests not installed)"

print(f"Data source : {data_source}")
print(f"Curve date  : {curve_date}")
display(curve)

Data source : FRED (live)
Curve date  : 2026-03-19 00:00:00


,T_years,y_decimal,P0T
0,0.083333,0.0373,0.996896
1,0.250000,0.0373,0.990718
2,0.500000,0.0376,0.981376
3,1.000000,0.0373,0.963387
4,2.000000,0.0379,0.927002
5,3.000000,0.0379,0.892526
6,5.000000,0.0388,0.823658
7,7.000000,0.0406,0.752616
8,10.000000,0.0425,0.653770
9,20.000000,0.0482,0.381364


## 2. Discount Factor Interpolator
Log-linear interpolation on the zero curve — same convention as the Hull-White model.

In [3]:
def build_df_interpolator(curve):
    """
    Build a log-linear discount factor interpolator from curve DataFrame.
    Returns a function: T -> P(0, T)
    """
    T_nodes = curve["T_years"].to_numpy(dtype=float)
    P_nodes = curve["P0T"].to_numpy(dtype=float)
    log_P   = np.log(np.clip(P_nodes, 1e-300, 1.0))
    interp  = interp1d(T_nodes, log_P, kind="linear",
                       fill_value="extrapolate", bounds_error=False)
    def df(T):
        return float(np.exp(interp(T)))
    return df


df_func = build_df_interpolator(curve)

# Sanity check
check = pd.DataFrame({
    "T_years": [1, 5, 10, 20, 30],
    "P(0,T)": [round(df_func(T), 6) for T in [1, 5, 10, 20, 30]]
})
print("Discount factor spot check:")
display(check)

Discount factor spot check:


,T_years,"P(0,T)"
0,1,0.963387
1,5,0.823658
2,10,0.653770
3,20,0.381364
4,30,0.234805


## 3. Loan Pool Definition

Hypothetical $100M 30-year fixed-rate RMBS pool. Parameters benchmarked to a 2024/2025 US residential mortgage pool.

**Justification of assumptions:**
- WAC 6.75%: Consistent with 30-year conforming mortgage rates, H2 2024
- WAM 360: Standard 30-year amortising pool, unseasoned
- Servicing spread 25bps: Market convention for agency-style pools

In [4]:
# ── Pool parameters (blue = inputs, change freely) ──────────────────────
POOL = {
    "balance":          100_000_000,   # $100M original pool balance
    "WAC":              0.0675,        # 6.75% weighted average coupon
    "WAM":              360,           # 360 months (30-year mortgages)
    "seasoning":        0,             # months already paid (0 = new)
    "servicing_spread": 0.0025,        # 25bps servicing fee p.a.
}

# Net coupon paid to investors after servicing
POOL["net_coupon"] = POOL["WAC"] - POOL["servicing_spread"]

# ── Default / loss assumptions ────────────────────────────────────────────
# CDR: Conditional Default Rate (annualised)
# Severity: Loss given default — % of defaulted balance lost
# Benchmarked to normalising US mortgage market, 2024-2025
DEFAULT = {
    "CDR":      0.015,   # 1.5% annual default rate
    "severity": 0.38,    # 38% loss severity (loss given default)
}

# ── Tranche structure ────────────────────────────────────────────────────
# Sequential pay: Senior absorbs first, Mezz second, Equity last
# Spreads over Treasury are market convention for each rating bucket
TRANCHES = pd.DataFrame([
    {"name": "Senior (AAA)",     "pct": 0.80, "spread": 0.0085},  # 85bps over Treasury
    {"name": "Mezzanine (BBB)",  "pct": 0.12, "spread": 0.0250},  # 250bps over Treasury
    {"name": "Equity / First Loss", "pct": 0.08, "spread": None}, # Residual — priced at IRR
])

TRANCHES["balance"] = TRANCHES["pct"] * POOL["balance"]

print("Pool summary:")
for k, v in POOL.items():
    print(f"  {k:<20}: {v}")
print("\nTranche structure:")
display(TRANCHES[["name", "pct", "balance", "spread"]])

Pool summary:
  balance             : 100000000
  WAC                 : 0.0675
  WAM                 : 360
  seasoning           : 0
  servicing_spread    : 0.0025
  net_coupon          : 0.065

Tranche structure:


,name,pct,balance,spread
0,Senior (AAA),0.80,80000000.0,0.0085
1,Mezzanine (BBB),0.12,12000000.0,0.0250
2,Equity / First Loss,0.08,8000000.0,NaN


## 4. PSA Prepayment Model

The PSA (Public Securities Association) benchmark is the industry-standard prepayment convention:
- **100 PSA**: CPR starts at 0.2% in month 1, increases 0.2% per month until month 30, then stays flat at 6% CPR
- **150 PSA**: All CPRs scaled by 1.5× (faster prepayment)
- **50 PSA**: All CPRs scaled by 0.5× (slower prepayment)

CPR is converted to SMM (Single Monthly Mortality) for month-by-month cash flow calculation:
$$SMM = 1 - (1 - CPR)^{1/12}$$

In [5]:
def psa_cpr_schedule(n_months, psa_speed=1.0, seasoning=0):
    """
    Generate monthly CPR schedule under PSA convention.

    Parameters
    ----------
    n_months  : total projection horizon in months
    psa_speed : PSA multiplier (1.0 = 100 PSA, 1.5 = 150 PSA, etc.)
    seasoning : months already paid on the pool

    Returns
    -------
    cpr : np.array of monthly CPR values
    smm : np.array of monthly SMM values
    """
    cpr = np.zeros(n_months)
    for t in range(1, n_months + 1):
        age = t + seasoning  # effective age of pool in month t
        if age <= 30:
            base_cpr = 0.06 * (age / 30)    # ramp: 0% → 6% over 30 months
        else:
            base_cpr = 0.06                  # flat at 6% CPR after month 30
        cpr[t - 1] = base_cpr * psa_speed

    smm = 1 - (1 - cpr) ** (1 / 12)         # convert annual CPR to monthly SMM
    return cpr, smm


# Preview 100 PSA schedule
cpr_100, smm_100 = psa_cpr_schedule(360, psa_speed=1.0)
preview = pd.DataFrame({
    "Month": [1, 6, 12, 24, 30, 36, 60, 120, 360],
    "CPR (100 PSA)": [round(cpr_100[t-1], 4) for t in [1,6,12,24,30,36,60,120,360]],
    "SMM (100 PSA)": [round(smm_100[t-1], 6) for t in [1,6,12,24,30,36,60,120,360]],
})
print("PSA prepayment schedule (100 PSA):")
display(preview)

PSA prepayment schedule (100 PSA):


,Month,CPR (100 PSA),SMM (100 PSA)
0,1,0.002,0.000167
1,6,0.012,0.001006
2,12,0.024,0.002022
3,24,0.048,0.004091
4,30,0.060,0.005143
5,36,0.060,0.005143
6,60,0.060,0.005143
7,120,0.060,0.005143
8,360,0.060,0.005143


## 5. Monthly Cash Flow Engine

For each month, the model calculates:
1. **Scheduled interest**: Net coupon × beginning balance / 12
2. **Scheduled principal**: From standard mortgage amortisation formula
3. **Prepayment**: SMM × (beginning balance − scheduled principal)
4. **Defaults**: MDR (monthly default rate) × remaining balance after prepayment
5. **Losses**: Defaults × severity (reduces recoverable balance)
6. **Total cash flow**: Interest + scheduled principal + prepayments + recoveries

$$MDR = 1 - (1 - CDR)^{1/12}$$

In [6]:
def generate_pool_cashflows(pool, default_params, psa_speed=1.0):
    """
    Generate monthly pool-level cash flows.

    Returns DataFrame with columns:
        month, beg_balance, scheduled_interest, scheduled_principal,
        prepayment, defaults, losses, recoveries, total_cf, end_balance
    """
    n        = pool["WAM"]
    r_m      = pool["net_coupon"] / 12          # monthly net rate
    r_gross  = pool["WAC"] / 12                 # monthly gross rate (for amortisation)
    B0       = pool["balance"]
    CDR      = default_params["CDR"]
    sev      = default_params["severity"]
    season   = pool["seasoning"]

    MDR = 1 - (1 - CDR) ** (1 / 12)            # monthly default rate

    cpr_sched, smm_sched = psa_cpr_schedule(n, psa_speed=psa_speed, seasoning=season)

    rows = []
    balance = B0

    for t in range(1, n + 1):
        if balance <= 0:
            break

        smm   = smm_sched[t - 1]
        rem   = n - (t - 1)                     # remaining periods

        # Scheduled interest on beginning balance
        interest = balance * r_m

        # Scheduled principal (mortgage amortisation formula)
        if r_gross > 0 and rem > 0:
            pmt         = balance * r_gross / (1 - (1 + r_gross) ** (-rem))
            sched_prin  = pmt - balance * r_gross
        else:
            sched_prin  = balance / max(rem, 1)

        sched_prin = min(sched_prin, balance)    # cap at remaining balance

        # Prepayment on balance after scheduled principal
        bal_after_sched  = balance - sched_prin
        prepayment       = smm * bal_after_sched

        # Defaults on remaining balance after prepayment
        bal_after_prep   = bal_after_sched - prepayment
        defaults         = MDR * bal_after_prep
        losses           = defaults * sev          # permanent loss
        recoveries       = defaults * (1 - sev)    # recovered as principal

        total_cf         = interest + sched_prin + prepayment + recoveries
        end_balance      = bal_after_prep - defaults

        rows.append({
            "month":               t,
            "beg_balance":         balance,
            "scheduled_interest":  interest,
            "scheduled_principal": sched_prin,
            "prepayment":          prepayment,
            "defaults":            defaults,
            "losses":              losses,
            "recoveries":          recoveries,
            "total_cf":            total_cf,
            "end_balance":         max(end_balance, 0),
        })

        balance = max(end_balance, 0)

    return pd.DataFrame(rows)


# Generate base case cash flows (100 PSA)
cf = generate_pool_cashflows(POOL, DEFAULT, psa_speed=1.0)

# Display summary
summary_months = [1, 6, 12, 24, 36, 60, 120, 240, 360]
display_cf = cf[cf["month"].isin(summary_months)][[
    "month", "beg_balance", "scheduled_interest",
    "scheduled_principal", "prepayment", "defaults", "losses", "total_cf"
]].copy()
for col in display_cf.columns[1:]:
    display_cf[col] = display_cf[col].round(0).astype(int)

print("Pool cash flows — base case (100 PSA, CDR 1.5%):")
display(display_cf)

print(f"\nTotal interest paid   : ${cf['scheduled_interest'].sum():,.0f}")
print(f"Total principal paid  : ${(cf['scheduled_principal'] + cf['prepayment'] + cf['recoveries']).sum():,.0f}")
print(f"Total losses          : ${cf['losses'].sum():,.0f}")
print(f"Loss as % of pool     : {cf['losses'].sum() / POOL['balance'] * 100:.2f}%")

Pool cash flows — base case (100 PSA, CDR 1.5%):


,month,beg_balance,scheduled_interest,scheduled_principal,prepayment,defaults,losses,total_cf
0,1,100000000,541667,86098,16668,125738,47781,722390
5,6,98691666,534580,87771,99150,123986,47115,798372
11,12,96586718,523178,89322,195151,121213,46061,882803
23,24,90754814,491589,90812,370888,113650,43187,1023752
35,36,83297235,451193,90272,427934,104192,39593,1033998
59,60,69427367,376065,88541,356610,86826,32994,875049
119,120,42958732,232693,84358,220503,53687,20401,570841
239,240,13223589,71628,76577,67615,16463,6256,226027
359,360,69513,377,69513,0,0,0,69889



Total interest paid   : $66,495,082
Total principal paid  : $94,168,906
Total losses          : $5,831,094
Loss as % of pool     : 5.83%


## 6. Sequential Pay Waterfall

Cash flows are distributed tranche by tranche in order of seniority:
1. **Senior** receives all principal cash flows until its balance is repaid, then passes to Mezzanine
2. **Mezzanine** receives remaining principal after Senior is paid off
3. **Equity** absorbs all residual cash flows and all credit losses

Interest is paid pro-rata to outstanding balance for Senior and Mezzanine. Equity receives residual interest.

In [7]:
def apply_waterfall(pool_cf, tranches):
    """
    Apply sequential pay waterfall to pool cash flows.

    Returns dict of DataFrames, one per tranche, with columns:
        month, interest_cf, principal_cf, loss_allocated, total_cf, balance
    """
    senior_bal = float(tranches.loc[tranches["name"].str.startswith("Senior"), "balance"].iloc[0])
    mezz_bal   = float(tranches.loc[tranches["name"].str.startswith("Mezz"),   "balance"].iloc[0])
    equity_bal = float(tranches.loc[tranches["name"].str.startswith("Equity"), "balance"].iloc[0])

    s_rows, m_rows, e_rows = [], [], []

    s_bal = senior_bal
    m_bal = mezz_bal
    e_bal = equity_bal

    pool_net_coupon = POOL["net_coupon"] / 12

    for _, row in pool_cf.iterrows():
        total_pool_bal = s_bal + m_bal + e_bal
        if total_pool_bal <= 0:
            break

        total_principal = (row["scheduled_principal"] +
                           row["prepayment"] +
                           row["recoveries"])
        total_loss      = row["losses"]
        total_interest  = row["scheduled_interest"]

        # ── Losses: equity absorbs first, then mezz, then senior ──────────
        loss_to_equity = min(total_loss, e_bal)
        loss_remainder = total_loss - loss_to_equity
        loss_to_mezz   = min(loss_remainder, m_bal)
        loss_remainder -= loss_to_mezz
        loss_to_senior = min(loss_remainder, s_bal)

        e_bal -= loss_to_equity
        m_bal -= loss_to_mezz
        s_bal -= loss_to_senior

        # ── Principal: sequential — senior first ──────────────────────────
        prin_to_senior = min(total_principal, s_bal)
        prin_remainder = total_principal - prin_to_senior
        prin_to_mezz   = min(prin_remainder, m_bal)
        prin_remainder -= prin_to_mezz
        prin_to_equity = min(prin_remainder, e_bal)

        s_bal -= prin_to_senior
        m_bal -= prin_to_mezz
        e_bal -= prin_to_equity

        # ── Interest: pro-rata to outstanding balance ────────────────────
        pool_bal_beg = row["beg_balance"]
        if pool_bal_beg > 0:
            int_to_senior = total_interest * (senior_bal / POOL["balance"])
            int_to_mezz   = total_interest * (mezz_bal   / POOL["balance"])
            int_to_equity = total_interest - int_to_senior - int_to_mezz
        else:
            int_to_senior = int_to_mezz = int_to_equity = 0.0

        month = int(row["month"])

        s_rows.append({"month": month,
                       "interest_cf":    int_to_senior,
                       "principal_cf":   prin_to_senior,
                       "loss_allocated": loss_to_senior,
                       "total_cf":       int_to_senior + prin_to_senior,
                       "balance":        max(s_bal, 0)})

        m_rows.append({"month": month,
                       "interest_cf":    int_to_mezz,
                       "principal_cf":   prin_to_mezz,
                       "loss_allocated": loss_to_mezz,
                       "total_cf":       int_to_mezz + prin_to_mezz,
                       "balance":        max(m_bal, 0)})

        e_rows.append({"month": month,
                       "interest_cf":    int_to_equity,
                       "principal_cf":   prin_to_equity,
                       "loss_allocated": loss_to_equity,
                       "total_cf":       int_to_equity + prin_to_equity,
                       "balance":        max(e_bal, 0)})

    return {
        "Senior":     pd.DataFrame(s_rows),
        "Mezzanine":  pd.DataFrame(m_rows),
        "Equity":     pd.DataFrame(e_rows),
    }


waterfall = apply_waterfall(cf, TRANCHES)

# Summary: total cash flows and losses per tranche
wf_summary = []
for name, df_t in waterfall.items():
    orig_bal = float(TRANCHES.loc[TRANCHES["name"].str.startswith(name), "balance"].iloc[0])
    wf_summary.append({
        "Tranche":          name,
        "Original Balance": orig_bal,
        "Total Interest CF":f"${df_t['interest_cf'].sum():,.0f}",
        "Total Principal CF":f"${df_t['principal_cf'].sum():,.0f}",
        "Total Losses":     f"${df_t['loss_allocated'].sum():,.0f}",
        "Loss Rate":        f"{df_t['loss_allocated'].sum() / orig_bal * 100:.2f}%",
    })

print("Waterfall summary — base case (100 PSA, CDR 1.5%):")
display(pd.DataFrame(wf_summary))

Waterfall summary — base case (100 PSA, CDR 1.5%):


,Tranche,Original Balance,Total Interest CF,Total Principal CF,Total Losses,Loss Rate
0,Senior,80000000.0,"$53,196,065","$80,000,000",$0,0.00%
1,Mezzanine,12000000.0,"$7,979,410","$12,000,000",$0,0.00%
2,Equity,8000000.0,"$5,319,607","$2,168,906","$5,831,094",72.89%


## 7. Tranche Pricing — Discounted Cash Flow

Each tranche is priced by discounting its cash flows at the risk-free rate plus tranche spread:
$$P = \sum_{t=1}^{T} \frac{CF_t}{(1 + r_t + s)^{t/12}}$$

Where $r_t$ is the interpolated Treasury yield for maturity $t/12$ and $s$ is the tranche spread.

**Weighted Average Life (WAL)** measures the effective duration of principal repayment:
$$WAL = \frac{\sum_t t \cdot P_t}{\sum_t P_t}$$
where $P_t$ is principal cash flow in month $t$.

In [8]:
def price_tranche(tranche_cf, orig_balance, spread, df_func):
    """
    Price a tranche by DCF using live Treasury curve + spread.

    Returns: price (% of par), WAL (years), yield (IRR), duration
    """
    pv = 0.0
    wal_num = 0.0
    total_prin = 0.0

    for _, row in tranche_cf.iterrows():
        t_years = row["month"] / 12.0
        cf      = row["total_cf"]
        prin    = row["principal_cf"]

        # Discount factor from live curve
        P0t   = df_func(t_years)

        # Spread-adjusted discount factor
        # Decompose: log P0t gives continuous zero rate; add spread
        if P0t > 0 and t_years > 0:
            zero_rate     = -np.log(P0t) / t_years
            disc_factor   = np.exp(-(zero_rate + spread) * t_years)
        else:
            disc_factor   = 1.0

        pv         += cf * disc_factor
        wal_num    += t_years * prin
        total_prin += prin

    price_pct = pv / orig_balance * 100  # price as % of par
    wal       = wal_num / total_prin if total_prin > 0 else 0.0

    # Compute IRR (yield) via Newton-Raphson
    cfs = [-orig_balance] + list(tranche_cf["total_cf"])
    months = [0] + list(tranche_cf["month"])

    def npv_at_rate(r_monthly):
        return sum(cf / (1 + r_monthly) ** (m) for cf, m in zip(cfs, months))

    # Bisect to find monthly IRR
    lo, hi = -0.01, 0.05
    for _ in range(100):
        mid = (lo + hi) / 2
        if npv_at_rate(mid) > 0:
            lo = mid
        else:
            hi = mid
    irr_monthly = (lo + hi) / 2
    irr_annual  = (1 + irr_monthly) ** 12 - 1

    return {
        "Price (% par)": round(price_pct, 3),
        "WAL (years)":   round(wal, 2),
        "Yield (IRR)":   round(irr_annual * 100, 3),
        "Spread (bps)":  round(spread * 10000, 0),
        "PV ($)": round(pv, 0),
    }


# Price Senior and Mezzanine at their spreads
spreads = {
    "Senior":    float(TRANCHES.loc[TRANCHES["name"].str.startswith("Senior"), "spread"].iloc[0]),
    "Mezzanine": float(TRANCHES.loc[TRANCHES["name"].str.startswith("Mezz"),   "spread"].iloc[0]),
}

orig_bals = {
    "Senior":    float(TRANCHES.loc[TRANCHES["name"].str.startswith("Senior"), "balance"].iloc[0]),
    "Mezzanine": float(TRANCHES.loc[TRANCHES["name"].str.startswith("Mezz"),   "balance"].iloc[0]),
    "Equity":    float(TRANCHES.loc[TRANCHES["name"].str.startswith("Equity"), "balance"].iloc[0]),
}

results = {}
for name in ["Senior", "Mezzanine"]:
    results[name] = price_tranche(
        waterfall[name], orig_bals[name], spreads[name], df_func
    )

# Equity: price at residual (spread = 0, priced at par, show IRR)
results["Equity"] = price_tranche(
    waterfall["Equity"], orig_bals["Equity"], 0.0, df_func
)

pricing_df = pd.DataFrame(results).T.reset_index()
pricing_df.columns = ["Tranche"] + list(pricing_df.columns[1:])

print("Tranche pricing — base case (100 PSA, CDR 1.5%):")
display(pricing_df)

Tranche pricing — base case (100 PSA, CDR 1.5%):


,Tranche,Price (% par),WAL (years),Yield (IRR),Spread (bps),PV ($)
0,Senior,115.216,7.98,7.678,85.0,92173057.0
1,Mezzanine,61.139,23.02,3.303,250.0,7336725.0
2,Equity,55.794,28.82,-0.464,0.0,4463501.0


## 8. Sensitivity Analysis

### Table 1: Senior Tranche Price (% par) — PSA Speed × CDR
Shows how prepayment risk (extension/contraction) and credit risk interact for the safest tranche.

### Table 2: Mezzanine Tranche Price (% par) — PSA Speed × CDR
Mezzanine is more exposed to credit risk — price falls sharply as defaults rise.

### Table 3: WAL (years) — Senior Tranche across PSA Speeds
Extension risk: slower prepayments = longer WAL = more duration risk.

In [9]:
psa_speeds = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
cdr_rates  = [0.005, 0.010, 0.015, 0.020, 0.030, 0.050]

# Build sensitivity grid
senior_price_grid = np.zeros((len(psa_speeds), len(cdr_rates)))
mezz_price_grid   = np.zeros((len(psa_speeds), len(cdr_rates)))
senior_wal_grid   = np.zeros((len(psa_speeds), len(cdr_rates)))

for i, psa in enumerate(psa_speeds):
    for j, cdr in enumerate(cdr_rates):
        d = {"CDR": cdr, "severity": DEFAULT["severity"]}
        cf_s = generate_pool_cashflows(POOL, d, psa_speed=psa)
        wf_s = apply_waterfall(cf_s, TRANCHES)

        res_s = price_tranche(wf_s["Senior"],    orig_bals["Senior"],    spreads["Senior"],    df_func)
        res_m = price_tranche(wf_s["Mezzanine"], orig_bals["Mezzanine"], spreads["Mezzanine"], df_func)

        senior_price_grid[i, j] = res_s["Price (% par)"]
        mezz_price_grid[i, j]   = res_m["Price (% par)"]
        senior_wal_grid[i, j]   = res_s["WAL (years)"]

cdr_labels = [f"{c*100:.1f}%" for c in cdr_rates]
psa_labels = [f"{int(p*100)} PSA" for p in psa_speeds]

df_senior_price = pd.DataFrame(senior_price_grid, index=psa_labels, columns=cdr_labels)
df_mezz_price   = pd.DataFrame(mezz_price_grid,   index=psa_labels, columns=cdr_labels)
df_senior_wal   = pd.DataFrame(senior_wal_grid,   index=psa_labels, columns=cdr_labels)

print("TABLE 1 — Senior Tranche Price (% par) | Rows: PSA Speed | Cols: CDR")
print("(Base case = 100 PSA, 1.5% CDR — highlighted)")
display(df_senior_price.round(2))

print("\nTABLE 2 — Mezzanine Tranche Price (% par) | Rows: PSA Speed | Cols: CDR")
display(df_mezz_price.round(2))

print("\nTABLE 3 — Senior Tranche WAL (years) | Rows: PSA Speed | Cols: CDR")
print("Note: WAL increases as prepayment slows (extension risk)")
display(df_senior_wal.round(2))

TABLE 1 — Senior Tranche Price (% par) | Rows: PSA Speed | Cols: CDR
(Base case = 100 PSA, 1.5% CDR — highlighted)


,0.5%,1.0%,1.5%,2.0%,3.0%,5.0%
50 PSA,116.73,115.62,114.56,113.56,111.69,108.40
75 PSA,117.01,115.99,115.02,114.10,112.37,109.31
100 PSA,117.04,116.10,115.22,114.37,112.78,109.97
125 PSA,116.84,115.99,115.18,114.41,112.96,110.39
150 PSA,116.47,115.70,114.97,114.27,112.95,110.61
200 PSA,115.46,114.84,114.24,113.66,112.58,110.63



TABLE 2 — Mezzanine Tranche Price (% par) | Rows: PSA Speed | Cols: CDR


,0.5%,1.0%,1.5%,2.0%,3.0%,5.0%
50 PSA,67.49,65.35,63.22,59.85,52.72,41.44
75 PSA,65.52,63.56,61.57,59.21,52.35,41.17
100 PSA,64.81,63.00,61.14,59.19,52.95,41.68
125 PSA,65.20,63.52,61.78,59.92,54.47,43.00
150 PSA,66.35,64.80,63.17,61.44,56.76,45.04
200 PSA,69.77,68.45,67.07,65.59,62.16,50.55



TABLE 3 — Senior Tranche WAL (years) | Rows: PSA Speed | Cols: CDR
Note: WAL increases as prepayment slows (extension risk)


,0.5%,1.0%,1.5%,2.0%,3.0%,5.0%
50 PSA,11.50,11.21,10.95,10.70,10.25,9.58
75 PSA,9.67,9.46,9.27,9.08,8.75,8.23
100 PSA,8.29,8.13,7.98,7.84,7.59,7.17
125 PSA,7.23,7.11,7.00,6.89,6.69,6.35
150 PSA,6.41,6.32,6.23,6.14,5.98,5.71
200 PSA,5.26,5.19,5.13,5.08,4.97,4.78


## 9. Spread Sensitivity — Break-Even Analysis

At what CDR does each tranche trade below par (price < 100)?
This identifies the **credit cushion** — the buffer before losses impair each tranche.

In [10]:
cdr_fine = np.linspace(0.001, 0.10, 50)   # CDR from 0.1% to 10%
senior_prices_fine = []
mezz_prices_fine   = []
equity_prices_fine = []

for cdr in cdr_fine:
    d    = {"CDR": cdr, "severity": DEFAULT["severity"]}
    cf_s = generate_pool_cashflows(POOL, d, psa_speed=1.0)
    wf_s = apply_waterfall(cf_s, TRANCHES)

    r_s = price_tranche(wf_s["Senior"],    orig_bals["Senior"],    spreads["Senior"],    df_func)
    r_m = price_tranche(wf_s["Mezzanine"], orig_bals["Mezzanine"], spreads["Mezzanine"], df_func)
    r_e = price_tranche(wf_s["Equity"],    orig_bals["Equity"],    0.0,                  df_func)

    senior_prices_fine.append(r_s["Price (% par)"])
    mezz_prices_fine.append(r_m["Price (% par)"])
    equity_prices_fine.append(r_e["Price (% par)"])

breakeven_df = pd.DataFrame({
    "CDR (%)": [round(c * 100, 2) for c in cdr_fine],
    "Senior Price (% par)": [round(p, 2) for p in senior_prices_fine],
    "Mezzanine Price (% par)": [round(p, 2) for p in mezz_prices_fine],
    "Equity Price (% par)": [round(p, 2) for p in equity_prices_fine],
})

# Find break-even CDR for each tranche (first CDR where price < 100)
def find_breakeven(prices, cdrs):
    for p, c in zip(prices, cdrs):
        if p < 100:
            return round(c * 100, 2)
    return ">10%"

be_senior = find_breakeven(senior_prices_fine, cdr_fine)
be_mezz   = find_breakeven(mezz_prices_fine,   cdr_fine)
be_equity = find_breakeven(equity_prices_fine, cdr_fine)

print("Break-even CDR (price falls below par):")
print(f"  Senior (AAA)    : {be_senior}% CDR")
print(f"  Mezzanine (BBB) : {be_mezz}% CDR")
print(f"  Equity          : {be_equity}% CDR")
print("\nBase case CDR: 1.5%")
print("\nBreak-even table (sample):")
display(breakeven_df.iloc[::5].reset_index(drop=True))

Break-even CDR (price falls below par):
  Senior (AAA)    : >10%% CDR
  Mezzanine (BBB) : 0.1% CDR
  Equity          : 0.1% CDR

Base case CDR: 1.5%

Break-even table (sample):


,CDR (%),Senior Price (% par),Mezzanine Price (% par),Equity Price (% par)
0,0.10,117.82,66.23,78.97
1,1.11,115.91,62.60,61.62
2,2.12,114.17,58.70,47.56
3,3.13,112.58,52.13,44.43
4,4.14,111.12,46.19,41.91
5,5.15,109.77,40.94,39.62
6,6.16,108.50,36.38,37.52
7,7.17,107.28,32.55,35.59
8,8.18,105.88,30.28,33.83
9,9.19,104.42,28.95,32.20


## 10. Model Summary & Key Outputs

In [11]:
print("=" * 65)
print("ABS PRICING MODEL — SUMMARY")
print("=" * 65)
print(f"Pool Balance         : $100,000,000")
print(f"WAC / Net Coupon     : {POOL['WAC']*100:.2f}% / {POOL['net_coupon']*100:.2f}%")
print(f"WAM                  : {POOL['WAM']} months (30-year)")
print(f"Prepayment model     : PSA (100 PSA base case)")
print(f"Default / Severity   : {DEFAULT['CDR']*100:.1f}% CDR / {DEFAULT['severity']*100:.0f}% severity")
print(f"Discount curve       : {data_source}")
print(f"Curve date           : {curve_date}")
print()
print("TRANCHE PRICING — BASE CASE")
print("-" * 65)
display(pricing_df)
print()
print("BREAK-EVEN CDR (price < par)")
print("-" * 65)
print(f"  Senior (AAA)     : {be_senior}% CDR  — {round(float(str(be_senior).replace('>',''))/DEFAULT['CDR']/100 if '>' not in str(be_senior) else 99, 1)}x credit cushion over base case")
print(f"  Mezzanine (BBB)  : {be_mezz}% CDR")
print(f"  Equity           : {be_equity}% CDR")
print()
print("KEY RISK OBSERVATIONS")
print("-" * 65)
print("1. Prepayment risk (PSA): Faster prepayment (200 PSA) shortens WAL")
print("   — contraction risk. Slower prepayment (50 PSA) extends WAL")
print("   — extension risk. Senior most exposed to extension.")
print("2. Credit risk (CDR): Equity absorbs first losses. Mezzanine price")
print("   deteriorates rapidly above ~3% CDR. Senior highly protected.")
print("3. Interest rate risk: Discount curve is live — model reprices")
print("   automatically when rates move. Higher rates = lower PV.")

ABS PRICING MODEL — SUMMARY
Pool Balance         : $100,000,000
WAC / Net Coupon     : 6.75% / 6.50%
WAM                  : 360 months (30-year)
Prepayment model     : PSA (100 PSA base case)
Default / Severity   : 1.5% CDR / 38% severity
Discount curve       : FRED (live)
Curve date           : 2026-03-19 00:00:00

TRANCHE PRICING — BASE CASE
-----------------------------------------------------------------


,Tranche,Price (% par),WAL (years),Yield (IRR),Spread (bps),PV ($)
0,Senior,115.216,7.98,7.678,85.0,92173057.0
1,Mezzanine,61.139,23.02,3.303,250.0,7336725.0
2,Equity,55.794,28.82,-0.464,0.0,4463501.0



BREAK-EVEN CDR (price < par)
-----------------------------------------------------------------
  Senior (AAA)     : >10%% CDR  — 99x credit cushion over base case
  Mezzanine (BBB)  : 0.1% CDR
  Equity           : 0.1% CDR

KEY RISK OBSERVATIONS
-----------------------------------------------------------------
1. Prepayment risk (PSA): Faster prepayment (200 PSA) shortens WAL
   — contraction risk. Slower prepayment (50 PSA) extends WAL
   — extension risk. Senior most exposed to extension.
2. Credit risk (CDR): Equity absorbs first losses. Mezzanine price
   deteriorates rapidly above ~3% CDR. Senior highly protected.
3. Interest rate risk: Discount curve is live — model reprices
   automatically when rates move. Higher rates = lower PV.
